# IPN-13 Temporal Factor V3 — calibration train/validation

Notebook này calibration `index_click_cycle`, `two_finger_click_cycle` và `pinch_opening/closing` trên **train + validation בלבד**. Không có cell, biến hoặc CLI option đọc test split.

Đầu ra là một ZIP gồm config được chọn bằng validation, metric từng cặp/subject, prediction từng mẫu, hashes và commit repo. Kết quả này chưa phải test accuracy và chưa bật pair-residual.

In [ ]:
from pathlib import Path
import datetime as dt
import hashlib
import importlib.util
import itertools
import json
import shutil
import subprocess
import sys
import uuid
import zipfile

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Không chạy trên Colab: bỏ qua mount Drive.')

# --- Chỉ sửa các biến trong khối này ---
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
REPO_REF = 'HuyenHoang'
REPO_COMMIT = None  # Khuyến nghị điền commit chứa temporal factor V3
EXPECTED_COUNTS = {'train': 2443, 'val': 589}
MIN_VALIDATION_COVERAGE_PER_PAIR = 0.25
SMOKE_TARGET_LIMIT = None  # Ví dụ 24; None = chạy full target classes

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:10]
WORK_ROOT = Path('/content') / f'ipn13_temporal_factor_v3_{RUN_ID}'
REPO_DIR = WORK_ROOT / 'repo'
LOCAL_OUTPUT = WORK_ROOT / 'calibration_output'
DRIVE_OUTPUT_ROOT = DRIVE_DATA_ROOT / 'outputs' / 'ipn13_temporal_factor_v3'
DRIVE_RUN_ROOT = DRIVE_DATA_ROOT / 'runs' / 'ipn13_temporal_factor_v3'
print({'run_id': RUN_ID, 'scope': 'train_validation_only', 'smoke_target_limit': SMOKE_TARGET_LIMIT})

## 1. Clone revision, cài dependency và kiểm tra code V3

Hãy commit/push các thay đổi V3 trước khi chạy. Tốt nhất điền `REPO_COMMIT` để artifact tái lập chính xác.

In [ ]:
COMMANDS = []
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def run_argv(argv, cwd=None, log_path=None):
    command = [str(value) for value in argv]
    COMMANDS.append(command)
    print('$', ' '.join(command))
    log = open(log_path, 'a', encoding='utf-8') if log_path else None
    try:
        process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            if log:
                log.write(line)
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, command)
    finally:
        if log:
            log.close()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

run_argv(['git', 'clone', '--filter=blob:none', REPO_URL, REPO_DIR])
run_argv(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_COMMIT or REPO_REF])
run_argv(['git', '-C', REPO_DIR, 'checkout', '--detach', REPO_COMMIT or 'FETCH_HEAD'])
REPO_COMMIT_RESOLVED = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, text=True, capture_output=True
).stdout.strip()
if REPO_COMMIT:
    assert REPO_COMMIT_RESOLVED == REPO_COMMIT, (REPO_COMMIT_RESOLVED, REPO_COMMIT)
if (REPO_DIR / 'requirements.txt').is_file():
    run_argv([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_DIR / 'requirements.txt'])
CALIBRATION_SCRIPT = REPO_DIR / 'scripts' / 'calibrate_ipn_temporal_factors.py'
FACTOR_MODULE = REPO_DIR / 'final' / 'code' / 'src_neurosymbolic' / 'predicates' / 'temporal_factors.py'
assert CALIBRATION_SCRIPT.is_file(), 'Thiếu calibration script: hãy commit và push code mới trước.'
assert FACTOR_MODULE.is_file(), 'Thiếu temporal_factors.py: hãy commit và push code mới trước.'
help_text = subprocess.run([sys.executable, str(CALIBRATION_SCRIPT), '--help'], check=True, text=True, capture_output=True).stdout
assert '--test' not in help_text.lower() and 'test_manifest' not in help_text.lower()
print('Repository revision:', REPO_COMMIT_RESOLVED)

## 2. Kiểm tra train/validation và tensor cache

Cell này không khai báo hoặc đọc test manifest.

In [ ]:
TRAIN_FILE = DRIVE_DATA_ROOT / 'train.txt'
VAL_FILE = DRIVE_DATA_ROOT / 'val.txt'
TRAIN_MANIFEST = DRIVE_DATA_ROOT / 'train_manifest.csv'
VAL_MANIFEST = DRIVE_DATA_ROOT / 'val_manifest.csv'
for path in (TRAIN_FILE, VAL_FILE, TRAIN_MANIFEST, VAL_MANIFEST):
    assert path.is_file(), f'Thiếu input: {path}'

def read_split(path):
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            source, _, label = line.rsplit(maxsplit=2)
            rows.append((source, int(label)))
    return rows

splits = {'train': read_split(TRAIN_FILE), 'val': read_split(VAL_FILE)}
for name, rows in splits.items():
    assert len(rows) == EXPECTED_COUNTS[name], (name, len(rows), EXPECTED_COUNTS[name])
    assert {label for _, label in rows} == set(range(13)), (name, sorted({label for _, label in rows}))
train_sources = {source.replace(chr(92), '/') for source, _ in splits['train']}
val_sources = {source.replace(chr(92), '/') for source, _ in splits['val']}
assert not (train_sources & val_sources), 'Train/validation source overlap'
CACHE_ROOT = next((path for path in (
    DRIVE_DATA_ROOT / 'keypoint_tensor_cache',
    DRIVE_DATA_ROOT / 'data' / 'keypoint_tensor_cache',
) if path.is_dir()), None)
assert CACHE_ROOT is not None, 'Không thấy keypoint_tensor_cache trên Drive.'
KEYPOINT_ROOT = DRIVE_DATA_ROOT / 'keypoints'
spec = importlib.util.spec_from_file_location('ipn_temporal_calibrator', CALIBRATION_SCRIPT)
calibrator = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = calibrator
spec.loader.exec_module(calibrator)
target_labels = {label for pair in calibrator.TARGET_PAIRS for label in pair}
preflight = {}
for name, split_file, manifest_file in (('train', TRAIN_FILE, TRAIN_MANIFEST), ('val', VAL_FILE, VAL_MANIFEST)):
    metadata = calibrator.manifest_labels(manifest_file)
    dataset = calibrator.KeypointSequenceDataset(split_file, KEYPOINT_ROOT, name, cache_root=CACHE_ROOT)
    resolved, missing = [], []
    for source, label_id in dataset.samples:
        calibrator.validate_source_split(source, name)
        label, _ = calibrator.label_from_source(source, name, metadata, label_id)
        if label not in target_labels:
            continue
        path = calibrator.resolve_sample_input(dataset, source)
        if path.suffix.lower() != '.pt' or not path.is_file():
            missing.append({'source': source, 'resolved': str(path)})
        else:
            resolved.append(path)
    assert not missing, f'{name}: thiếu tensor cache cho target samples; ví dụ {missing[:10]}'
    preflight[name] = len(resolved)
print({'counts': {name: len(rows) for name, rows in splits.items()}, 'cache_root': str(CACHE_ROOT), 'target_cache_preflight': preflight})

## 3. Tạo grid threshold

Grid kết hợp ba mức click, hai mức đồng bộ hai ngón và ba mức pinch — tổng cộng 18 cấu hình. Train chỉ dùng để diagnostics; script chọn duy nhất bằng validation objective với coverage gate.

In [ ]:
click_settings = [
    {'click_active_threshold': .50, 'click_inactive_threshold': .42, 'click_min_confidence': .30, 'temporal_max_gap': 2},
    {'click_active_threshold': .56, 'click_inactive_threshold': .46, 'click_min_confidence': .35, 'temporal_max_gap': 3},
    {'click_active_threshold': .62, 'click_inactive_threshold': .50, 'click_min_confidence': .45, 'temporal_max_gap': 4},
]
sync_settings = [
    {'two_finger_start_delta': 3, 'two_finger_overlap': .50},
    {'two_finger_start_delta': 5, 'two_finger_overlap': .35},
]
pinch_settings = [
    {'pinch_min_excursion': .08, 'pinch_min_valid_ratio': .50, 'pinch_dominance_ratio': 1.10},
    {'pinch_min_excursion': .12, 'pinch_min_valid_ratio': .60, 'pinch_dominance_ratio': 1.20},
    {'pinch_min_excursion': .16, 'pinch_min_valid_ratio': .70, 'pinch_dominance_ratio': 1.35},
]
GRID = [{**click, **sync, **pinch} for click, sync, pinch in itertools.product(click_settings, sync_settings, pinch_settings)]
GRID_PATH = WORK_ROOT / 'temporal_factor_grid.json'
GRID_PATH.write_text(json.dumps(GRID, indent=2, sort_keys=True), encoding='utf-8')
print('Grid configs:', len(GRID), '->', GRID_PATH)

## 4. Chạy calibration

`SMOKE_TARGET_LIMIT` giới hạn số mẫu thuộc sáu lớp mục tiêu trên mỗi split, không phải N dòng đầu manifest. Đặt `None` để chạy đầy đủ.

In [ ]:
argv = [
    sys.executable, '-u', CALIBRATION_SCRIPT,
    '--data-root', DRIVE_DATA_ROOT,
    '--keypoint-root', KEYPOINT_ROOT,
    '--cache-root', CACHE_ROOT,
    '--grid', GRID_PATH,
    '--output-dir', LOCAL_OUTPUT,
    '--min-coverage', MIN_VALIDATION_COVERAGE_PER_PAIR,
]
if SMOKE_TARGET_LIMIT is not None:
    argv += ['--limit', SMOKE_TARGET_LIMIT]
run_argv(argv, log_path=WORK_ROOT / 'calibration.log')
assert (LOCAL_OUTPUT / 'calibration_report.json').is_file()
assert (LOCAL_OUTPUT / 'selected_config.json').is_file()
assert (LOCAL_OUTPUT / 'input_tensors.jsonl').is_file()

## 5. Xem kết quả validation

`overall_accuracy_abstain_wrong` tính abstention là sai. Chỉ config đạt coverage tối thiểu ở từng cặp mới được xem là feasible.

In [ ]:
report = json.loads((LOCAL_OUTPUT / 'calibration_report.json').read_text(encoding='utf-8'))
selected = report['selected_validation_score']
print('Selected config index:', report['selected_config_index'])
print(json.dumps(selected['config'], indent=2, sort_keys=True))
print({'support': selected['support'], 'coverage': selected['coverage'],
       'accuracy_abstain_wrong': selected['overall_accuracy_abstain_wrong'], 'feasible': selected['feasible']})
for pair_name, metrics in selected['per_pair'].items():
    print(pair_name, {key: metrics[key] for key in (
        'support', 'coverage', 'accuracy_on_covered', 'overall_accuracy_abstain_wrong')})
    print(json.dumps(metrics['confusion_unknown_included'], indent=2))
if not selected['feasible']:
    print('WARNING: Không config nào thỏa coverage gate; không dùng selected_config để tích hợp pair-residual.')

## 6. Đóng gói artifact lên Drive

ZIP này là file cần gửi lại để audit. Nó không chứa checkpoint và không chứa kết quả test.

In [ ]:
import torch
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
drive_output = DRIVE_OUTPUT_ROOT / RUN_ID
run_zip = DRIVE_RUN_ROOT / f'{RUN_ID}.zip'
if drive_output.exists() or run_zip.exists():
    raise FileExistsError(f'Từ chối ghi đè: {drive_output} hoặc {run_zip}')
shutil.copytree(LOCAL_OUTPUT, drive_output)
shutil.copy2(GRID_PATH, drive_output / 'temporal_factor_grid.json')
shutil.copy2(WORK_ROOT / 'calibration.log', drive_output / 'calibration.log')
notebook_manifest = {
    'run_id': RUN_ID,
    'created_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'scope': 'train_validation_only',
    'repo_commit': REPO_COMMIT_RESOLVED,
    'commands': COMMANDS,
    'inputs': {str(path): sha256_file(path) for path in (TRAIN_FILE, VAL_FILE, TRAIN_MANIFEST, VAL_MANIFEST)},
    'grid_sha256': sha256_file(GRID_PATH),
    'smoke_target_limit': SMOKE_TARGET_LIMIT,
    'min_validation_coverage_per_pair': MIN_VALIDATION_COVERAGE_PER_PAIR,
    'torch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
(drive_output / 'notebook_manifest.json').write_text(json.dumps(notebook_manifest, indent=2), encoding='utf-8')
with zipfile.ZipFile(run_zip, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(drive_output.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(drive_output))
print('Calibration ZIP:', run_zip)
print('SHA-256:', sha256_file(run_zip))

## 7. Cách đọc và bước tiếp theo

- Nếu `feasible=false`, giữ V3 tắt và gửi ZIP để audit detector/coverage.
- Nếu feasible nhưng accuracy-on-covered thấp, chưa triển khai pair-residual.
- Nếu cả coverage và pair accuracy đủ tốt trên validation, freeze config rồi mới xây pair-residual trên neural validation predictions.
- Không mở test và không sửa threshold dựa trên các confusion test đã biết.